# LGN phosphene simulator: a guided tour

`dynaphos_lgn` simulates what a patient would see when electrodes in the **lateral geniculate nucleus (LGN)** are stimulated. Unlike the cortical Dynaphos simulator, you do not choose where the phosphenes appear. Each electrode sits in a voxel of a reconstructed macaque LGN atlas (Erwin et al., 1999), and the atlas decides the phosphene's position, size and shape.

**Contents**

1. Build a simulator and render one frame
2. One nucleus or two
3. Stimulation parameters: current and pulse width
4. Model parameters: the constant K and the magnification model
5. Placing electrodes yourself
6. Inside the simulator
7. From camera image to phosphenes
8. Optimising the stimulation

Throughout, the simulator covers **both** nuclei, and so the whole visual field. `demo_simulator_lgn.ipynb` has the long version of every section, including a switch to run everything with a single nucleus.

> **Treat the pictures as a working model, not as measurements.** Almost nothing about LGN stimulation has been measured. The current-spread constant is borrowed from macaque V1, the temporal dynamics from human V1, and the right nucleus is the left one mirrored. `dynaphos_lgn/README.md` lists the known gaps.

## 0. Setup

Every simulator in this notebook is built by `make_simulator`. It loads the shipped config, applies a handful of settings, and calls `build_simulator`, the library's single entry point. `render` stimulates every electrode at the same current and returns one frame.

In [1]:
import sys
sys.path.append('./..')

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch

from dynaphos import utils
from dynaphos.image_processing import sobel_processor
from dynaphos.utils import to_numpy

from dynaphos_lgn.build import build_simulator

CONFIG = '../config/params_lgn.yaml'
ATLAS_DIR = '../data/Erwin_Atlas'      # the .DAT files and JACOBIAN.npz
COLOURS = {'left': 'tab:blue', 'right': 'tab:red'}


def load_config(**overrides):
    """The shipped config, with `section__key=value` overrides.

    e.g. load_config(current_spread__k_ua_per_mm2=6750)
    """
    params = utils.load_params(CONFIG)
    params['atlas']['hemispheres'] = ['left', 'right']    # whole field
    # params['electrodes']['max_eccentricity_deg'] = 8.0    # fill the frame
    params['run']['fps'] = 10                             # 1 frame = 100 ms
    params['magnification']['model'] = 'anisotropic_gradient'
    for key, value in overrides.items():
        section, name = key.split('__')
        params[section][name] = value
    return params


def make_simulator(n_electrodes=100, **overrides):
    """Build a simulator with `n_electrodes` in EACH nucleus."""
    params = load_config(electrodes__n_electrodes=n_electrodes, **overrides)
    return build_simulator(params, atlas_dir=ATLAS_DIR)


def render(simulator, current_ua=100.):
    """One frame with every electrode at `current_ua`."""
    simulator.reset()
    return simulator(current_ua * 1e-6 * torch.ones(simulator.num_phosphenes))


HALF = load_config()['run']['view_angle'] / 2
EXTENT = [-HALF, HALF, -HALF, HALF]            # image axes in degrees


def show(ax, image, title=''):
    """A rendered frame, with axes in degrees of visual angle."""
    ax.imshow(to_numpy(image), cmap='gray', origin='lower',
              vmin=0, vmax=1, extent=EXTENT)
    ax.set(title=title, xlabel='x (deg)', ylabel='y (deg)')

## 1. Build a simulator and render one frame

`build_simulator` does the whole chain: load the atlas, place the electrodes, look up each phosphene's position and size, and return a ready simulator. With two nuclei it returns a `BilateralLGNSimulator`, which holds one simulator per nucleus and adds their images together.

By default the electrodes are spread evenly over the visual field. `n_electrodes` counts **per nucleus**, and every per-electrode vector lists the left nucleus first.

In [2]:
simulator, parts = make_simulator(n_electrodes=100)

print(type(simulator).__name__, '-', simulator.num_phosphenes, 'electrodes')
for name, span in simulator.electrode_slices.items():
    print(f'  {name:<5} nucleus: electrodes {span.start}-{span.stop - 1}')

BilateralLGNSimulator - 200 electrodes
  left  nucleus: electrodes 0-99
  right nucleus: electrodes 100-199


The simulator takes one current per electrode (in amperes) and returns an image. `reset()` clears its memory of earlier frames.

In [3]:
amplitude = 50e-6   # amperes
stim = amplitude * torch.ones(simulator.num_phosphenes)
simulator.reset()
frame = simulator(stim)

fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))
xy = simulator.vf_xy
for name, span in simulator.electrode_slices.items():
    axs[0].plot(*xy[span].T, '.', color=COLOURS[name], label=f'{name} LGN')
axs[0].set(xlim=EXTENT[:2], ylim=EXTENT[2:], aspect='equal',
           xlabel='x (deg)', ylabel='y (deg)', title='Phosphene positions')
axs[0].legend(fontsize=8)
show(axs[1], frame, f'Simulated percept, {int(amplitude * 1e6)} uA')
fig.tight_layout()

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

Each nucleus fills one half of the field: the left LGN sees the right half, and the right LGN the left half. At this current the phosphenes are far larger than the gaps between them, so they merge into one bright blob. Sections 3 and 4 look at why.

## 2. One nucleus or two

One LGN represents only the opposite half of the visual field. The atlas is a single left nucleus, so on its own it covers the right half of the field only. The right nucleus is built by mirroring the left one.

Which nuclei are simulated is one config key, `atlas.hemispheres`. It can also be passed straight to `build_simulator`:

In [ ]:
one_nucleus, _ = build_simulator(load_config(electrodes__n_electrodes=100),
                                 atlas_dir=ATLAS_DIR, hemispheres=['left'])

fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))
show(axs[0], render(one_nucleus), "hemispheres=['left']")
show(axs[1], frame, "hemispheres=['left', 'right']")
fig.tight_layout()

With one nucleus the left half of every frame stays black. Keep in mind that the two halves are **not independent**: they are the same tissue reflected, so any quirk of the atlas appears in both, mirrored.

## 3. Stimulation parameters

Each electrode is driven by three numbers: amplitude, pulse width and frequency. Each can be one value per electrode, or a single number for all of them. This section sweeps them on the 100-per-nucleus array from section 1.

#### 3a. Current: size against spacing

Two electrodes look like two separate spots only while they are more than about **2σ apart** (Sparrow's criterion). The spacing is fixed by where the electrodes are, but σ depends on the current: less current recruits less tissue, so the phosphene is smaller. It is also dimmer, and below threshold it is not seen at all.

In [ ]:
AMPLITUDES_UA = [30., 50., 70., 100.]   # max_current_ua in the config is 100

xy = simulator.vf_xy
distance = np.linalg.norm(xy[:, None] - xy[None, :], axis=-1)
np.fill_diagonal(distance, np.inf)
spacing = distance.min(axis=1)          # nearest-neighbour spacing, deg

fig, axs = plt.subplots(1, len(AMPLITUDES_UA) + 1, figsize=(19, 4))
sigmas = []
print(f'{"uA":>5} {"sigma (deg)":>12} {"resolvable":>11} {"detected":>9}')
for ax, amplitude in zip(axs, AMPLITUDES_UA):
    image = render(simulator, amplitude)
    major, _ = simulator.phosphene_sigma_deg(amplitude)
    detected = to_numpy(simulator.get_state()['detection_probability']).ravel()
    resolvable = 100 * np.mean(spacing > 2 * major)
    sigmas.append(np.nanmedian(major))
    show(ax, image, f'{amplitude:.0f} uA')
    print(f'{amplitude:5.0f} {sigmas[-1]:12.2f} {resolvable:10.0f}% '
          f'{100 * np.mean(detected > 0.5):8.0f}%')

axs[-1].plot(AMPLITUDES_UA, sigmas, 'o-', label='median sigma')
axs[-1].axhline(np.median(spacing) / 2, color='k', ls='--',
                label='half the median spacing')
axs[-1].set(xlabel='current (uA)', ylabel='deg', title='Size vs. spacing')
axs[-1].legend(fontsize=8)
fig.tight_layout()
print(f'\nMedian spacing {np.median(spacing):.2f} deg. "resolvable": nearest '
      f'neighbour > 2 sigma away. "detected": p(detect) > 0.5.')

**Lowering the current does not separate them.** At 100 µA only 4% of electrodes are resolvable. At 30 µA that rises to 32%, but by then no phosphene is reliably seen. The electrodes are 0.90° apart, and σ never drops below half of that before the phosphenes fade out. With this array, current alone cannot make the phosphenes distinct.

#### 3b. Pulse width: brightness without size

Longer pulses deliver more charge per second, so the phosphenes get brighter. Their size is set by the amplitude alone, so it stays the same.

In [ ]:
PULSE_WIDTHS = [170e-6, 400e-6, 800e-6]      # seconds

fig, axs = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, pw in zip(axs, PULSE_WIDTHS):
    simulator.reset()
    image = simulator(35e-6 * torch.ones(simulator.num_phosphenes),
                      pw, 300.)                 # amplitude, pulse width, Hz
    state = simulator.get_state()
    show(ax, image, f'35 uA, 300 Hz, {pw * 1e6:.0f} us\n'
                    f'brightness {to_numpy(state["brightness"]).mean():.2f}, '
                    f'sigma {np.nanmedian(to_numpy(state["sigma_major"])):.2f} deg')
fig.tight_layout()

Frequency works the same way. The temporal model is carried over unchanged from V1: there is no strength-duration data for LGN at all, so the pulse-width dependence is borrowed, not measured.

## 4. Model parameters

Section 3 varied how the electrodes are driven. This section varies two assumptions inside the model instead. Both change phosphene size, and neither is well constrained for LGN. It uses 25 electrodes per nucleus so the individual phosphenes can be told apart.

#### 4a. The excitability constant K

In the model, σ scales as `sqrt(current / K)`. K = 675 µA/mm² is borrowed from macaque V1 and has never been measured in LGN; it is the least certain number in the model. The electrodes are placed before any current flows, so changing K leaves them where they are and changes only the phosphene size. The values below are a factor of ten apart and are not proposals. They show how much of the blur this one borrowed constant controls.

In [ ]:
K_VALUES = [675., 6_750., 67_500., 675_000.]    # 675 is the shipped value

sparse, _ = make_simulator(n_electrodes=25)      # shipped K, reused in 4b

fig, axs = plt.subplots(1, len(K_VALUES), figsize=(17, 4.2))
for ax, k in zip(axs, K_VALUES):
    sim_k = sparse if k == 675. else make_simulator(
        n_electrodes=25, current_spread__k_ua_per_mm2=k)[0]
    major, _ = sim_k.phosphene_sigma_deg(100.)
    show(ax, render(sim_k), f'K = {k:,.0f} uA/mm^2\n'
                            f'median sigma {np.nanmedian(major):.2f} deg')
fig.tight_layout()

At the shipped K the phosphenes are large, soft blobs. Ten times larger K already separates them, and shows that each phosphene is an **ellipse**, not a disc. The atlas stretches visual space differently in different directions, and each ellipse's elongation and orientation are read from the tissue at its electrode.

The simulated sizes also run 4–6× larger than an independent estimate (Vurro et al., 2014). So read any "how many electrodes are usable" answer as a property of this K, not of the LGN.

#### 4b. The magnification model

"Magnification" converts millimetres of tissue into degrees of visual field, and it sets how large each phosphene comes out. `magnification.model` selects one of three:

| model | source | shape |
|---|---|---|
| `jacobian` (default) | each electrode's own voxel in the fitted atlas | ellipse |
| `anisotropic_gradient` | the same field, pre-binned into a small (eccentricity, inclination) table | ellipse |
| `malpeli_density` | Malpeli et al. (1996)'s closed-form cell-density formula | circle |

The electrodes, current and current spread are the same for all three. Only the magnification changes.

In [ ]:
models = {'jacobian': sparse}               # the default, built in 4a
for model in ['anisotropic_gradient', 'malpeli_density']:
    models[model], _ = make_simulator(n_electrodes=25,
                                      magnification__model=model)

fig, axs = plt.subplots(1, 3, figsize=(14, 4.5))
print(f'{"model":<22}{"sigma major":>12}{"sigma minor":>12}   (deg, median)')
for ax, (name, sim) in zip(axs, models.items()):
    major, minor = sim.phosphene_sigma_deg(100.)
    print(f'{name:<22}{np.nanmedian(major):>12.2f}{np.nanmedian(minor):>12.2f}')
    show(ax, render(sim), name)
fig.tight_layout()

The two Jacobian-based models agree closely; the table model is cheaper to build. The Malpeli model has to be circular and is a check on overall magnitude, not on shape. It is not independent evidence, though: the atlas was itself reconstructed from Malpeli's data. The long notebook compares the models per electrode and by eccentricity.

## 5. Placing electrodes yourself

There are two ways to place electrodes by hand instead of spreading them automatically.

#### 5a. By position in the tissue

`voxel_indices` takes atlas voxels, one entry per nucleus. Below, the same 6 × 6 grid (0.4 mm pitch) goes into each nucleus. Distance is measured from the midline, so the same coordinates land on mirror-image tissue.

In [ ]:
atlases = {name: part['atlas'] for name, part in parts['hemispheres'].items()}

# Centre the grid on the atlas's representation of the central 5 degrees.
left = atlases['left']
central = np.argwhere(left.valid & ~left.is_ipsi_sentinel()
                      & (left.eccentricity_deg <= 5))
ml0, dv0, ap0 = left.index_to_horsley_clarke(*central.mean(axis=0))
offsets = (np.arange(6) - 2.5) * 0.4                     # mm
ml, dv = np.meshgrid(ml0 + offsets, dv0 + offsets, indexing='ij')

voxels = {}
for name, atlas in atlases.items():
    index = np.stack(atlas.horsley_clarke_to_index(
        ml.ravel(), dv.ravel(), np.full(ml.size, ap0), hemisphere=name), -1)
    # Drop grid points that miss the reconstructed tissue.
    usable = (atlas.valid[tuple(index.T)]
              & ~atlas.is_ipsi_sentinel(atlas.inclination[tuple(index.T)]))
    voxels[name] = index[usable]
    print(f'{name:<5} LGN: {usable.sum()} of {usable.size} grid points on tissue')

grid_sim, _ = build_simulator(load_config(), atlas_dir=ATLAS_DIR,
                              voxel_indices=voxels)

# A coronal slice of the left nucleus, coloured by layer, with the grid on it
k = voxels['left'][0, 2]
layers = np.where(left.valid[:, :, k], left.layer[:, :, k], np.nan)
rows, cols = np.nonzero(left.valid[:, :, k])

fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))
axs[0].imshow(layers, origin='lower', cmap='tab10', vmin=0.5, vmax=10.5,
              interpolation='nearest')
axs[0].plot(voxels['left'][:, 1], voxels['left'][:, 0], 'w.', markersize=6)
axs[0].set(xlim=(cols.min() - 10, cols.max() + 10),
           ylim=(rows.min() - 10, rows.max() + 10),
           xlabel='dorsal-ventral (voxels, 25 um)',
           ylabel='medial-lateral (voxels)',
           title='Left LGN: layers and the electrode grid')
xy = grid_sim.vf_xy
for name, span in grid_sim.electrode_slices.items():
    axs[1].plot(*xy[span].T, '.', color=COLOURS[name], label=f'{name} LGN')
axs[1].set(xlim=EXTENT[:2], ylim=EXTENT[2:], aspect='equal',
           xlabel='x (deg)', ylabel='y (deg)', title='Where the grid lands')
axs[1].legend(fontsize=8)
show(axs[2], render(grid_sim, 80.), 'Percept, 80 uA')
fig.tight_layout()

A regular grid in the tissue is **not** regular in the visual field. It is dense near the fovea, sparse further out, and folded where it crosses a layer boundary. The coloured bands are the atlas's layers: two magnocellular and two parvocellular codes, one per eye.

#### 5b. By position in the visual field

`visual_field_targets` works the other way round. You give (eccentricity, inclination) positions in degrees, and each electrode goes to the voxel that best matches one of them. Each target is sent to the nucleus that covers its half of the field. The match is the nearest voxel, so `position_error_deg` reports how far off each one landed.

In [ ]:
eccentricity = np.array([1., 2., 3., 4., 5., 6., 2., 4.])
inclination = np.array([-75., -15., 45., 135., 175., -135., -170., 100.])

target_sim, target_parts = build_simulator(
    load_config(), atlas_dir=ATLAS_DIR,
    visual_field_targets=(eccentricity, inclination))

for name, part in target_parts['hemispheres'].items():
    errors = part['electrode_array'].position_error_deg
    print(f'{name:<5} LGN: {len(errors)} targets, miss '
          + ', '.join(f'{e:.2f}' for e in errors) + ' deg')

fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))
requested = eccentricity * np.stack([np.cos(np.deg2rad(inclination)),
                                     np.sin(np.deg2rad(inclination))])
axs[0].plot(*requested, 'ko', fillstyle='none', markersize=9, label='requested')
xy = target_sim.vf_xy
for name, span in target_sim.electrode_slices.items():
    axs[0].plot(*xy[span].T, '.', color=COLOURS[name], label=f'{name} LGN')
axs[0].set(xlim=EXTENT[:2], ylim=EXTENT[2:], aspect='equal',
           xlabel='x (deg)', ylabel='y (deg)', title='Requested vs. placed')
axs[0].legend(fontsize=8)
show(axs[1], render(target_sim, 80.), 'Percept, 80 uA')
fig.tight_layout()

## 6. Inside the simulator

`get_state()` returns every internal quantity, one row per electrode, left nucleus first. Two entries are specific to LGN:

- `sigma_major` / `sigma_minor`: the two axes of each phosphene ellipse, in degrees.
- `layer_activation`: how much of each LGN layer the current reached, which `layer_activation_table()` breaks down by layer.

A small array (3 electrodes per nucleus) keeps the output readable.

In [ ]:
small, _ = make_simulator(n_electrodes=3)
small.reset()
small(100e-6 * torch.rand(small.num_phosphenes))

state = small.get_state()
print('get_state() keys:', ', '.join(state))
print('electrode    :', list(small.hemisphere_of_electrode))
print('brightness   :', np.round(to_numpy(state['brightness']).ravel(), 3))
print('sigma major  :', np.round(to_numpy(state['sigma_major']).ravel(), 2))
print('sigma minor  :', np.round(to_numpy(state['sigma_minor']).ravel(), 2))

fig, ax = plt.subplots(figsize=(7, 3.5))
bottom = np.zeros(small.num_phosphenes)
for layer, values in small.layer_activation_table().items():
    values = np.ravel(values)
    ax.bar(range(small.num_phosphenes), values, bottom=bottom, label=layer)
    bottom += values
ax.axvline(small.electrode_slices['right'].start - 0.5, color='k', ls='--')
ax.set(xlabel='electrode (left | right nucleus)',
       ylabel='recruitment', title='Which layers each electrode reached')
ax.legend(fontsize=8)

Each nucleus keeps its own state and its own per-electrode thresholds, reachable as `simulator.simulators['left']` and `simulator.simulators['right']`. A different number of electrodes per side is set with `electrodes.n_electrodes_by_hemisphere`.

## 7. From camera image to phosphenes

An image becomes a percept in three steps:

1. **Simplify** the image, e.g. with edge detection.
2. **Sample** it at each electrode's receptive field (`sample_stimulus`) to get one current per electrode.
3. **Simulate** the percept from those currents.

Each nucleus samples the whole image, and because their electrodes cover opposite halves of the field, together they read all of it.

In [ ]:
image_sim, _ = make_simulator(n_electrodes=150)

img = cv2.imread('../test/data/tudelft_logo.png')[::-1]          # origin lower-left
gray = 255 - cv2.cvtColor(cv2.resize(img, (256, 256)), cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (21, 21), 7)
inputs = {'grayscale': gray, 'Sobel edges': sobel_processor(blur)}

fig, axs = plt.subplots(2, 2, figsize=(9, 8.5))
for column, (name, picture) in enumerate(inputs.items()):
    axs[0, column].imshow(picture, cmap='gray', origin='lower', extent=EXTENT)
    axs[0, column].set_title(f'input: {name}')
    # Normalise to [0, 1]; rescale=True then maps 1 onto 100 uA
    stim = image_sim.sample_stimulus(picture / picture.max(), rescale=True)
    image_sim.reset()
    show(axs[1, column], image_sim(stim), 'percept')
fig.suptitle(f'{image_sim.num_phosphenes} electrodes '
             f'({image_sim.num_phosphenes // 2} per nucleus)')
fig.tight_layout()

The filled logo saturates into one blob, while the edge map keeps the ring and the stroke. That is why edge detection is the usual preprocessing step. The electrodes reach 8° from the centre, the edge of the frame, so only the four corners of the image are never read.

**Do not forget `rescale=True`.** Without it, the sampled intensities (0 to 1) are read as amperes, so a white pixel becomes 1 A, ten thousand times the intended 100 µA. No warning is raised:

In [ ]:
print(f"without rescale: max {image_sim.sample_stimulus(gray).max():.1e} A")
print(f"rescale=True   : max {image_sim.sample_stimulus(gray, rescale=True).max():.1e} A")

## 8. Optimising the stimulation

Section 7 used a **fixed rule**: each electrode gets the brightest pixel in its receptive field, scaled to 100 µA, at the default 170 µs and 300 Hz. Nothing checks whether the percept that comes out looks like the image.

The simulator is written in PyTorch and every step is differentiable, so the currents can instead be **learned**. (The full end-to-end optimisation used in the paper lives in [this repository](https://github.com/neuralcodinglab/viseon/tree/dynaphos-paper).)

#### 8a. Matching an image

We render the percept, compare it with the edge map (mean squared error, pixel by pixel), and let gradient descent adjust every electrode's current. Two details:

- The current is written as `100 µA × sigmoid(x)`, so it can never leave the 0–100 µA range the simulator was built for.
- Optimisation starts from the fixed rule's currents, raised to at least 30 µA. Below the rheobase (23.9 µA) no charge reaches the tissue, so an electrode starting there would get no gradient and never move.

In [ ]:
MAX_CURRENT = 100e-6
RHEOBASE = 23.9e-6            # thresholding.rheobase: below it, nothing is seen
edges = inputs['Sobel edges'] / inputs['Sobel edges'].max()
target = torch.tensor(edges, dtype=torch.float32)


def mse(sim, stim):
    """Render `stim` and compare the percept with the target."""
    sim.reset()
    percept = sim(stim)
    return torch.nn.functional.mse_loss(percept, target), percept


def optimise(sim, steps=60):
    """Learn one current per electrode so the percept matches `target`."""
    fixed = sim.sample_stimulus(edges, rescale=True)
    start = fixed.clamp(30e-6, 0.95 * MAX_CURRENT) / MAX_CURRENT
    x = torch.nn.Parameter(torch.log(start / (1 - start)))   # sigmoid(x) = start
    optimizer = torch.optim.Adam([x], lr=0.1)
    losses = []
    for _ in range(steps):                                    # ~1 s per step
        optimizer.zero_grad()
        loss, _ = mse(sim, MAX_CURRENT * torch.sigmoid(x))
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return fixed, (MAX_CURRENT * torch.sigmoid(x)).detach(), losses


fixed_stim, learned_stim, losses = optimise(image_sim)
fixed_loss, fixed_percept = mse(image_sim, fixed_stim)
fixed_loss = fixed_loss.item()
_, learned_percept = mse(image_sim, learned_stim)

fig, axs = plt.subplots(1, 4, figsize=(19, 4.5))
show(axs[0], target, 'target: Sobel edges')
show(axs[1], fixed_percept, f'fixed rule\nMSE {fixed_loss:.3f}')
show(axs[2], learned_percept, f'optimised currents\nMSE {losses[-1]:.3f}')
axs[3].plot(losses, label='optimised')
axs[3].axhline(fixed_loss, color='k', ls='--', label='fixed rule')
axs[3].set(xlabel='iteration', ylabel='MSE', title='Loss')
axs[3].legend(fontsize=8)
fig.tight_layout()

for name, stim in [('fixed rule', fixed_stim), ('optimised', learned_stim)]:
    on = stim[stim > RHEOBASE]
    print(f'{name:<11}: {len(on)} of {len(stim)} electrodes above rheobase, '
          f'median {on.median() * 1e6:.0f} uA, '
          f'{(on > 90e-6).float().mean() * 100:.0f}% of them above 90 uA')

The optimised percept matches the edge map about three times as well (MSE 0.126 → 0.040). It gets there mostly by turning currents **down**. The same number of electrodes stay above the rheobase (84), but their median current drops from 63 to 50 µA. Smaller, dimmer phosphenes overlap less, so the ring and the two strokes appear as bands rather than solid blobs. The optimiser has found the trade-off from section 3a by itself: at this spacing, less current means less merging.

This is still a toy. The loss compares pixels and the currents are fitted to this one image. A practical encoder would be a network trained over many images. The loss also need not be on the image: any quantity from `get_state()`, such as per-electrode brightness, works the same way.

#### 8b. What the gradients reach

The two nuclei are summed into one image, so one loss should reach the currents of both. A quick check:

In [ ]:
image_sim.reset()
probe = torch.full((image_sim.num_phosphenes,), 80e-6, requires_grad=True)
image_sim(probe).sum().backward()
for name, span in image_sim.electrode_slices.items():
    print(f'{name:<5} LGN: |gradient| = {probe.grad[span].abs().sum():.3e}')

Gradients reach amplitude, pulse width and frequency. They do not reach electrode positions, which are fixed at build time.

#### 8c. Soft or hard detection threshold

The percept is each phosphene's brightness times its **probability of being detected**. That probability compares the tissue activation with a per-electrode threshold, and `thresholding.soft_threshold` decides how:

- `True` (the default): a smooth logistic curve, as measured in detection experiments. It starts at 0 at zero activation and rises steadily.
- `False`: Dynaphos's original hard step, where a phosphene is either fully there or not there at all.

First, the curves themselves, for the three left-nucleus electrodes of the small array from section 6, all driven at the same current. The thresholds are drawn from the same seed, so both versions use the same thresholds.

In [ ]:
small_hard, _ = make_simulator(n_electrodes=3,
                               thresholding__soft_threshold=False)
currents = np.arange(0., 101., 1.)

fig, axs = plt.subplots(1, 2, figsize=(13, 4.2))
for (gate, sim), style in zip({'soft': small, 'hard': small_hard}.items(),
                              ['-', '--']):
    p_detect, gradient = [], []
    for ua in currents:
        stim = torch.full((sim.num_phosphenes,), ua * 1e-6, requires_grad=True)
        sim.reset()
        sim(stim).sum().backward()
        p_detect.append(sim.get_state()['detection_probability']
                        .detach().numpy().ravel()[:3])
        gradient.append(stim.grad.numpy()[:3])
    for i in range(3):
        axs[0].plot(currents, np.array(p_detect)[:, i], style, color=f'C{i}',
                    label=f'electrode {i}, {gate}')
        axs[1].plot(currents, np.array(gradient)[:, i], style, color=f'C{i}')
for ax in axs:
    ax.axvline(RHEOBASE * 1e6, color='grey', lw=0.8)
    ax.set_xlabel('current (uA)')
axs[0].set(ylabel='p(detect)', title='Detection probability')
axs[0].legend(fontsize=7, ncol=2)
axs[1].set(ylabel='d percept / d current',
           title="Gradient of each phosphene w.r.t. its own current")
fig.tight_layout()

Both gates stay at zero below the rheobase (grey line), where no charge reaches the tissue. Above it they differ:

- **Hard:** each electrode stays at 0 until its own threshold, then jumps to 1. Between the rheobase and that threshold the gradient is exactly **zero**, so an optimiser cannot tell that more current would help.
- **Soft:** detection and gradient rise from the rheobase on, so every electrode above it gets a signal. The price is that the soft curve is more cautious: it reaches full detection later than the step does.

What that means in practice: the same image optimisation as in 8a, once with each gate.

In [ ]:
image_hard, _ = make_simulator(n_electrodes=150,
                               thresholding__soft_threshold=False)
fixed_hard, learned_hard, losses_hard = optimise(image_hard)
fixed_loss_hard = mse(image_hard, fixed_hard)[0].item()
_, learned_percept_hard = mse(image_hard, learned_hard)

fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))
for gate, curve, colour in [('soft', losses, 'C0'), ('hard', losses_hard, 'C1')]:
    axs[0].plot(curve, color=colour, label=f'{gate}: optimised')
axs[0].axhline(fixed_loss, color='C0', ls=':', label='soft: fixed rule')
axs[0].axhline(fixed_loss_hard, color='C1', ls=':', label='hard: fixed rule')
axs[0].set(xlabel='iteration', ylabel='MSE', title='Loss')
axs[0].legend(fontsize=8)
show(axs[1], learned_percept, f'soft threshold\nMSE {losses[-1]:.3f}')
show(axs[2], learned_percept_hard, f'hard threshold\nMSE {losses_hard[-1]:.3f}')
fig.tight_layout()

for gate, sim, stim in [('soft', image_sim, learned_stim),
                        ('hard', image_hard, learned_hard)]:
    sim.reset()
    sim(stim)
    detected = to_numpy(sim.get_state()['detection_probability']).ravel() > 0.5
    print(f'{gate}: {int((stim > RHEOBASE).sum())} of {len(stim)} electrodes '
          f'above rheobase, {detected.sum()} of them detected')

With the hard gate the optimisation stalls at MSE 0.063, against 0.040 with the soft one. The printout shows why. 262 electrodes are still above the rheobase, but only 29 are detected. The rest are sitting below their own threshold, where the hard gate passes no gradient, so they never moved from where they started. With the soft gate, 46 are detected.

Note also that the fixed rule scores differently under the two gates (MSE 0.126 soft, 0.141 hard) even though the currents are identical. The switch changes the **model**, not just the optimiser, so say which one you used when reporting results.

---

## Summary

- **The atlas sets the phosphenes.** Electrode position in the LGN fixes where each phosphene appears, how large it is, and its elliptical shape.
- **One nucleus covers half the field.** The whole field needs both (`atlas.hemispheres: [left, right]`), and the second nucleus is a mirror image of the first.
- **At the shipped parameters, phosphenes are large relative to their spacing.** Lowering the current barely helps and pulse width only changes brightness. The size depends mostly on K, the least certain number in the model.
- **Electrodes can be placed three ways:** spread automatically, placed by tissue position, or placed by visual-field target.
- **Everything is differentiable** with respect to the stimulation parameters. Learning the currents against an image gives a clearer percept than sampling the image directly.
- **The soft detection threshold is what makes that work.** With the hard step, electrodes below their threshold get no gradient and the optimisation stalls.